In [4]:
from datasets import load_dataset
dataset = load_dataset("lhoestq/conll2003")

Generating test split: 100%|██████████| 3453/3453 [00:00<00:00, 711132.85 examples/s]


In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [6]:
dataset['train'][0]

{'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

In [11]:
#tagnames=dataset['train'].features['ner_tags'].feature.names

In [12]:
#print(type(dataset['train'].features['ner_tags']))

In [13]:
#len(tagnames)

Masking Token = Padding 
Sentence lengths different hote hain.

Sentence1: EU rejects German call
Sentence2: British lamb

Padding ke baad:

EU rejects German call
British lamb <PAD> <PAD>

Agar masking na karo to model PAD tokens pe bhi loss compute karega, jo galat hai.

In [14]:
from collections import Counter
counter=Counter()
#training set se vocabulary learn krte h
for example in dataset['train']:
    counter.update(token.lower() for token in example ['tokens'])
vocab={}
#KUCH ESE WORDS JO TRAIN M ARE BUT TEST M NHI UNKE LIE SPCL TOKEN
vocab={'PAD':0,'UNK':1}
for word in counter:
    vocab[word]=len(vocab)


In [15]:
vocab

{'PAD': 0,
 'UNK': 1,
 'eu': 2,
 'rejects': 3,
 'german': 4,
 'call': 5,
 'to': 6,
 'boycott': 7,
 'british': 8,
 'lamb': 9,
 '.': 10,
 'peter': 11,
 'blackburn': 12,
 'brussels': 13,
 '1996-08-22': 14,
 'the': 15,
 'european': 16,
 'commission': 17,
 'said': 18,
 'on': 19,
 'thursday': 20,
 'it': 21,
 'disagreed': 22,
 'with': 23,
 'advice': 24,
 'consumers': 25,
 'shun': 26,
 'until': 27,
 'scientists': 28,
 'determine': 29,
 'whether': 30,
 'mad': 31,
 'cow': 32,
 'disease': 33,
 'can': 34,
 'be': 35,
 'transmitted': 36,
 'sheep': 37,
 'germany': 38,
 "'s": 39,
 'representative': 40,
 'union': 41,
 'veterinary': 42,
 'committee': 43,
 'werner': 44,
 'zwingmann': 45,
 'wednesday': 46,
 'should': 47,
 'buy': 48,
 'sheepmeat': 49,
 'from': 50,
 'countries': 51,
 'other': 52,
 'than': 53,
 'britain': 54,
 'scientific': 55,
 'was': 56,
 'clearer': 57,
 '"': 58,
 'we': 59,
 'do': 60,
 "n't": 61,
 'support': 62,
 'any': 63,
 'such': 64,
 'recommendation': 65,
 'because': 66,
 'see': 67,
 '

In [16]:
len(vocab)

21011

In [17]:
def encode(example):
    input_ids = [vocab.get(token.lower(), 1) for token in example['tokens']]
    
    return {
        'input_ids': input_ids,
        'labels': example['ner_tags']
    }

In [18]:
dataset=dataset.map(encode)

Map: 100%|██████████| 3453/3453 [00:00<00:00, 5395.16 examples/s] 


In [19]:
dataset['train'][0]

{'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0],
 'input_ids': [2, 3, 4, 5, 6, 7, 8, 9, 10],
 'labels': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

#what is to be applies collectively on the entire batch
#dyanamic padding = jo max length h batch ki sabki len utni kar do
#rather than fix krne se 256
#Collate function kya hota hai?

#Collate function DataLoader ko batata hai ki multiple samples ko ek batch me kaise combine karna hai.

Kyuki sentences ki length different hoti hai, unko same length ka banana padta hai.
def collate_fn(batch):
    

In [20]:
def collate_fn(batch):
    maxlen=([len(x['input_ids']) for x in batch])
    input_ids=[]
    labels=[]
    mask=[]
    for x in batch:
        L=len(x['input_ids'])
        pad_length=maxlen-L
        input_ids.append(x['input_ids']+[0]*pad_length)
        labels.append(x['labels']+[0]*pad_length)
        mask.append([1]*L+[0]*pad_length)
    return torch.tensor(input_ids),torch.tensor(labels),torch.tensor(mask)
    

In [22]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    dataset['train'],
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    dataset['test'],
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

In [ ]:
#scratch LSTM
# embedding layer--lstm--output layer
import torch.nn as nn

class LSTMModel(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_tags):
        super(LSTMModel, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.hidden_dim

        #input gate
        self.Wix=nn.Linear(embedding_dim, hidden_dim)
        self.Wia=nn.Linear(hidden_dim, hidden_dim)
        #output gate
        self.Wox=nn.Linear(embedding_dim, hidden_dim)
        self.Woa=nn.Linear(hidden_dim, hidden_dim)
        #forgrt gate
        self.Wfx=nn.Linear(embedding_dim, hidden_dim)
        self.Wfa=nn.Linear(hidden_dim, hidden_dim)
         #candidate gate
        self.Wcx=nn.Linear(embedding_dim, hidden_dim)
        self.Wca=nn.Linear(hidden_dim, hidden_dim)
        #outputlayer
        self.fc=nn.Linear(hidden_dim,num_tags)
        
    def forward(self,input,mask):
        emb=self.embedding(input)
        #s=variable dyanamic padding
        B,T,d=emb.shape
        h=torch.zeros(B,self.hidden_dim,device=input.device)
        c=torch.zeros(B,self.hidden_dim,device=input.device)
        
        
        for t in range(T):
            xt=emb[:,t,:]
            #a =torch.cat(xt,h)
            i=torch.sigmoid(self.W_ix(xt)+self.W_ia(h))
            f=torch.sigmoid(self.W_fx(xt)+self.W_fa(h))
            o=torch.sigmoid(self.W_ox(xt)+self.W_oa(h))
            cand=torch.tanh(self.W_cx(xt)+self.W_ca(h))
            c=i*cand+f*c
            h=o*torch.tanh(C)
            h=h*mask[:,t].unsqueeze(1)
            c=c*mask[:,t].unsqueeze(1)
            output=self.fc(h)
            #pytorch m cross entropy m hi softmax hota h
            outputs.append(output)
        
        
        

        

    
    
        
    

In [23]:
import torch
import torch.nn as nn

class LSTMModel(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_tags):
        super(LSTMModel, self).__init__()

        # hidden dimension store karna
        self.hidden_dim = hidden_dim

        # word ids -> word vectors
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # input gate
        self.Wix = nn.Linear(embedding_dim, hidden_dim)
        self.Wih = nn.Linear(hidden_dim, hidden_dim)

        # output gate
        self.Wox = nn.Linear(embedding_dim, hidden_dim)
        self.Woh = nn.Linear(hidden_dim, hidden_dim)

        # forget gate
        self.Wfx = nn.Linear(embedding_dim, hidden_dim)
        self.Wfh = nn.Linear(hidden_dim, hidden_dim)

        # candidate state
        self.Wcx = nn.Linear(embedding_dim, hidden_dim)
        self.Wch = nn.Linear(hidden_dim, hidden_dim)

        # final layer (NER tags prediction)
        self.fc = nn.Linear(hidden_dim, num_tags)


    def forward(self, input, mask):

        # input ids -> embeddings
        emb = self.embedding(input)

        # B=batch size, T=seq length, d=embedding dim
        B, T, d = emb.shape

        # initial hidden state
        h = torch.zeros(B, self.hidden_dim, device=input.device)

        # initial cell state
        c = torch.zeros(B, self.hidden_dim, device=input.device)

        outputs = []

        # sequence loop (time steps)
        for t in range(T):

            # current word embedding
            xt = emb[:, t, :]

            # input gate
            i = torch.sigmoid(self.Wix(xt) + self.Wih(h))

            # forget gate
            f = torch.sigmoid(self.Wfx(xt) + self.Wfh(h))

            # output gate
            o = torch.sigmoid(self.Wox(xt) + self.Woh(h))

            # candidate cell state
            cand = torch.tanh(self.Wcx(xt) + self.Wch(h))

            # cell state update
            c = f * c + i * cand

            # hidden state
            h = o * torch.tanh(c)

            # masking (padding tokens ignore karne ke liye)
            h = h * mask[:, t].unsqueeze(1)
            c = c * mask[:, t].unsqueeze(1)

            # tag prediction
            output = self.fc(h)

            outputs.append(output)

        

        return torch.stack(outputs,dim=1)

In [25]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = LSTMModel(len(vocab), 128, 128, 9).to(device)

In [ ]:
def masked_loss(otputs,labels,mask):
    b,T,c=outputs.shape
    outputs=outputs.view(B*T,c)
    labels=labels.view(B*T)
    mask=mask.view(B*T)
    loss=torch.nn.functional.cross_entropy(outputs,labels)
    loss=loss*mask
    return loss.sum()/mask.sum()
    
    